# Generating h3 Hexgrids from GeoDataFrames

In [1]:
import os

import matplotlib.pyplot as plt

from tobler.area_weighted import area_interpolate
from tobler.util import h3fy

%load_ext watermark
%watermark -v -a "author: eli knaap" -d -u -p tobler,cenpy,geopandas

Author: author: eli knaap

Last updated: 2026-10-31

Python implementation: CPython
Python version       : 3.14.10
IPython version      : 9.18.0

tobler   : 0.14.1.dev83+gfb0403854.d20261031
cenpy    : 1.0.1
geopandas: 1.2.0



**Note**: This notebook relies on functionality from the [`contextily`](https://contextily.readthedocs.io/en/latest/) package that provides convenient basemaps for geospatial plots, and the [`cenpy`](https://cenpy-devs.github.io/cenpy/) package that provide a convenient interface to the U.S. Census API. These can be installed with  
  
`pip install contextily cenpy`     
or    
`conda install contextily cenpy -c conda-forge`

In [2]:
from cenpy import products, set_sitekey

## Getting data from CenPy

To begin with, we will fetch some data from the 2017 ACS

In [3]:
set_sitekey(os.environ["CENSUS_API_KEY"])  # you need to provide your own API key

acs = products.ACS(2017)

KeyError: 'CENSUS_API_KEY'

We're looking for median home value, so first we will filter the ACS tables by those containing "value" in the description so we can find the correct variable code

In [ ]:
acs.filter_tables("VALUE", by="description")

The variable we're looking for is `B25077_001E`, the median home value of each. Lets collect that data for the Washington DC metropolitan region. The next cell can take a minute or two to run, depending on the speed of your connection.

In [ ]:
dc = acs.from_msa("Washington-Arlington", variables=["B25077_001E"])

In [ ]:
dc.info()

In [ ]:
dc.crs

## Creating Hexgrids with the `h3fy` function

Using the `h3fy` function from the `tobler.util` module, we can easily generate a hexgrid covering the face of the DC Metropolitan region

In [ ]:
dc_hex = h3fy(dc)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 10))
axs = axs.flatten()

dc.plot(ax=axs[0], alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)
dc_hex.plot(ax=axs[1], alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)

axs[0].set_title("Original Tract Data")
axs[1].set_title("Hex Grid")

for ax in axs.flat:
    ax.axis("off")

By altering the `resolution` parameter, we can generate grids using hexes of various sizes

In [ ]:
dc_hex_large = h3fy(dc, resolution=5)
dc_hex_small = h3fy(dc, resolution=7)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 10))

dc_hex_large.plot(ax=axs[0], alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)
dc_hex_small.plot(ax=axs[1], alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)

for ax in axs.flat:
    ax.axis("off")

and by using the `clip` parameter, we can ensure that the hexgrid is does not extend beyond the borders of the input geodataframe

In [ ]:
dc_hex_clipped = h3fy(dc, resolution=5, clip=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

dc_hex_clipped.plot(ax=ax, alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)
ax.axis("off")
ax.set_title("h3fy with clip")

Notice, though, that we still have some sharp hexagonal edges, meaning the hexagon doesn't *fully* cover the original source polygons. To force full coverage, we can use the `buffer` keyword (or, optionally, both `buffer` and `clip`).

In [ ]:
dc_hex_clipped = h3fy(dc, resolution=5, clip=True, buffer=True)

fig, ax = plt.subplots(figsize=(10, 10))

dc_hex_clipped.plot(ax=ax, alpha=0.4, linewidth=1.6, edgecolor="white", tiles=True)
ax.axis("off")
ax.set_title("h3fy with buffer & clip")

## Interpolating to a hexgrid

Thus, in just a few lines of code, we can estimate the value of census variables represented by a regular hexgrid

here, we will estimate the median home value of each hex in the DC region using simple areal interpolation

In [ ]:
dc_hex_interpolated = area_interpolate(
    source_df=dc, target_df=dc_hex, intensive_variables=["B25077_001E"], fill_nan="mean"
)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 10))
dc.plot("B25077_001E", scheme="quantiles", alpha=0.5, ax=axs[0], tiles=True)

dc_hex_interpolated.plot(
    "B25077_001E",
    scheme="quantiles",
    alpha=0.5,
    ax=axs[1],
    tiles=True,
)

axs[0].set_title("Original Data")
axs[1].set_title("Interpolated Data")

for ax in axs:
    ax.axis("off")
plt.suptitle(
    r"Spatial Interpolation with the PySAL $\mathtt{tobler}$ package", fontsize=16
)

## Dealing with NaN Values

It's important to understand how and NaN (not-a-number) values (i.e. missing data) in the source dataframe are handled. 

In [ ]:
fill_nans = [None, 0.0, "mean", "median"]

f, ax = plt.subplots(2, 2, figsize=(12, 12))
ax = ax.flatten()
for i, nan in enumerate(fill_nans):
    dc_hex_interpolated = area_interpolate(
        source_df=dc,
        target_df=dc_hex,
        intensive_variables=["B25077_001E"],
        fill_nan=nan,
    ).plot("B25077_001E", scheme="quantiles", alpha=0.5, ax=ax[i])
    ax[i].set_title(f"fill_nan={nan}")

Further, all of these options are **still** different from dropping any *geometries* with missing values for the variable being interpolated. In this case, the dropped observations in `source` are not part of the overlay system so only the overlapping pieces of `source` and `target` count toward the total weight applied.

In [ ]:
ax = dc_hex_interpolated = area_interpolate(
    source_df=dc.dropna(subset=["B25077_001E"]),
    target_df=dc_hex,
    intensive_variables=["B25077_001E"],
    fill_nan=nan,
).plot("B25077_001E", scheme="quantiles", alpha=0.5)
ax.set_title("drop NA prior to interpolation")
ax.axis("off")

For extensive variables (where target values are a weighted sum), dropping missing observations is functionally equivalent to setting `fill_nan=0` and `allocate_total=True`. 

For intensive variables, however, the weighted average provided by each `target` zone is not "diluted" by missing data. The total weights at the target polygon is the shared area of source and target, not the total area of target.